In [3]:
import os ,json,time
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from pageindex import PageIndexClient
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:qwen/qwen3-32b")

In [5]:
llm.invoke("hi")

AIMessage(content='<think>\nOkay, the user said "hi". That\'s pretty open-ended. I should respond in a friendly and welcoming way. Maybe ask how I can assist them. Keep it simple and let them know I\'m here to help with whatever they need. No need for any complicated steps here since it\'s just a greeting. Just a straightforward, positive reply.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 9, 'total_tokens': 95, 'completion_time': 0.196359214, 'completion_tokens_details': None, 'prompt_time': 0.000364355, 'prompt_tokens_details': None, 'queue_time': 0.54788421, 'total_time': 0.196723569}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f01f0-a844-70f0-bea9-a31e0e48aba2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'o

In [11]:
PDF_PATH="../data/pdf_files/2024-wttc-introduction-to-ai.pdf"

In [12]:
pi_client = PageIndexClient(api_key=os.environ.get("PAGEINDEX_API_KEY"))

In [13]:
print(f"uploading {PDF_PATH} to PageIndex...")

uploading ../data/pdf_files/2024-wttc-introduction-to-ai.pdf to PageIndex...


In [14]:
result=pi_client.submit_document(PDF_PATH)
doc_id=result["doc_id"]

In [15]:
doc_id

'pi-cmqudj5mc00a401pebj3fu5lf'

In [18]:
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result["status"].lower()

    print(f"status: {status}")

    if status in ("ready", "completed"):
        print("Document is ready!")
        break

    elif status in ("failed", "error"):
        print("Document processing failed")
        break

    time.sleep(5)

status: completed
Document is ready!


In [19]:
tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get("result",[])
print(f"top level section {len(pageindex_tree)}")
print("raw tree first node")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {},indent=2))

top level section 1
raw tree first node
{
  "title": "INTRODUCTION",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# INTRODUCTION\nTO ARTIFICIAL\nINTELLIGENCE (AI)\nTECHNOLOGY\n\nGUIDE FOR TRAVEL & TOURISM LEADERS\n\nJanuary 2024\n",
  "text": "# INTRODUCTION\nTO ARTIFICIAL\nINTELLIGENCE (AI)\nTECHNOLOGY\n\nGUIDE FOR TRAVEL & TOURISM LEADERS\n\nJanuary 2024\n",
  "nodes": [
    {
      "title": "CONTENTS",
      "node_id": "0001",
      "page_index": 2,
      "summary": "## CONTENTS\n\nFOREWORD ... 3\n\nINTRODUCTION ... 4\n\nBrief history of Artificial Intelligence (AI) ... 4\n\nWhat is Artificial Intelligence & Why There is Global Interest Now ... 7\n\nAlgorithms : The Brains of AI ... 8\n\nData : The Fuel That Drives AI ... 12\n\nComputing Power : The Machines Behind AI ... 16\n\nTypes of Artificial Intelligence ... 21\n\nGenerative AI ... 24\n\nGlobal Digital Divide & AI Skills Gap ... 31\n\nQUIZ ... 34\n\nANNEX : AI TERMINOLOGY ... 35\n\nQUIZ ANSWER ... 40\n\nACKNOWL

In [25]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("-- " if indent > 0 else "")
        page = node.get("page_index", node.get("page", "?"))

        print(f"{prefix}[{node['node_id']}] {node['title']} (page {page})")

        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full document structure")
print_tree(pageindex_tree)

Full document structure
[0000] INTRODUCTION (page 1)
  -- [0001] CONTENTS (page 2)
  -- [0002] FOREWORD (page 3)
  -- [0003] INTRODUCTION (page 4)
    -- [0004] BRIEF HISTORY OF ARTIFICIAL INTELLIGENCE (AI) (page 4)
    -- [0005] WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW (page 7)
    -- [0006] ALGORITHMS : THE BRAINS OF AI (page 8)
    -- [0007] DATA : THE FUEL THAT DRIVES AI (page 12)
    -- [0008] COMPUTING POWER : THE MACHINES BEHIND AI (page 16)
    -- [0009] TYPES OF ARTIFICIAL INTELLIGENCE (page 21)
    -- [0010] GENERATIVE AI (page 24)
    -- [0011] GLOBAL DIGITAL DIVIDE & AI SKILLS GAP (page 31)
  -- [0012] QUIZ (page 34)
  -- [0013] ANNEX : AI TERMINOLOGY (page 35)
  -- [0014] QUIZ ANSWER (page 40)
  -- [0015] ACKNOWLEDGEMENTS (page 41)
  -- [0016] ENDNOTES (page 42)


In [26]:
def count_nodes(nodes):
    total=len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total
total=count_nodes(pageindex_tree)
print(f"Total nodes in document: {total}")
print(" Each node * one reterivable section of the document")

Total nodes in document: 17
 Each node * one reterivable section of the document


In [33]:
import json

def llm_tree_search(query: str, tree: list, llm, max_results: int = 3):
    """
    Search the tree for relevant nodes based on the query using the LLM.
    Returns a dict containing thinking and node_list.
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page_index": n.get("page_index", n.get("page", "?")),
                "summary": n.get("summary", ""),
            }

            if n.get("nodes"):
                entry["nodes"] = compress(n["nodes"])

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""
You are given a query and a document's tree structure (like a Table of Contents).

Your task is to identify which node IDs most likely contain the answer.

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this JSON format:

{{
    "thinking": "<brief reasoning>",
    "node_list": [1, 2]
}}
"""

    response = llm.invoke(prompt)

    content = response.content.strip()

    # Remove markdown if model returns ```json
    if content.startswith("```"):
        content = content.replace("```json", "").replace("```", "").strip()

    return json.loads(content)

In [35]:
query = "What is covered in this file?"

result = llm_tree_search(query, pageindex_tree, llm)

print("LLM Reasoning:")
print(result["thinking"])

print("Selected Node IDs:")
print(result["node_list"])

LLM Reasoning:
The query 'What is covered in this file?' is most likely to be answered by the table of contents or an introductory section. Node '0001' with the title 'CONTENTS' seems to be a direct match as it contains a detailed list of the file's contents. Additionally, node '0000' with the title 'INTRODUCTION' could also be relevant as it may provide an overview of the file's coverage.
Selected Node IDs:
['0001', '0000']


In [36]:
def find_nodes_by_id(tree:list,target_ids:list)->list:
    """recurisvely walk the tree and collect nodes whose node_id is in target_ids"""
    found=[]
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_id(node["nodes"], target_ids))
    return found

In [ ]:
def generate_answer(query: str, nodes: list, llm) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return " No relevant sections found in the document."

    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.

Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.

Be concise and precise.

Question:
{query}

Context:
{context}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [44]:
def vectorless_rag(query: str, tree: list, llm, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree, llm)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_id(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes, llm)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

In [45]:
answer = vectorless_rag(
    query="What are the syllabus covered in this file",
    tree=pageindex_tree,
    llm=llm
)

🔍 Query: What are the syllabus covered in this file

🧠 Reasoning: The query asks about the syllabus covered in the file, which is most likely to be found in the table of contents or introduction sections. Node 0001 contains the table of contents, which lists the var...
🎯 Retrieved node IDs: ['0001']
📄 Sections found: ['CONTENTS']

📝 Answer:
The syllabus covered in this file includes: 
1. Brief history of Artificial Intelligence (AI) 
2. What is Artificial Intelligence & Why There is Global Interest Now 
3. Algorithms 
4. Data 
5. Computing Power 
6. Types of Artificial Intelligence 
7. Generative AI 
8. Global Digital Divide & AI Skills Gap (CONTENTS | 2)
